Qtd de dutos
tlen de dutos
tlen do ac (separado por tipo de vala)
tlen do dc

In [4]:
from DWG_to_Excel_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - BOQ.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - BOQ.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - BOQ.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [5]:
df = pd.DataFrame(data)

print(df.describe())
print(df.info())
print(df['layer'].value_counts())

            start_x        start_y         end_x          end_y        length
count  40737.000000   40737.000000  40737.000000   40737.000000  40737.000000
mean   35061.879632  128095.237106  35061.916944  128104.782502     54.851697
std      597.953849     958.361380    597.325738     957.024467     53.722982
min    33667.550000  125994.798675  33669.246800  125994.798675      0.000000
25%    34616.394850  127314.491400  34620.053650  127272.127575      6.300000
50%    35003.193200  128067.624300  34996.957868  128067.624300     44.631731
75%    35492.963200  128981.706640  35502.053650  128995.651570     92.256585
max    36625.890000  129808.228288  36624.193200  129708.943463    454.309533
<class 'pandas.DataFrame'>
RangeIndex: 40737 entries, 0 to 40736
Data columns (total 8 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   type     40737 non-null  str    
 1   layer    40737 non-null  str    
 2   handle   40737 non-null  str    
 3   start_

In [6]:
mask_ac_cables = (df['layer'] == "EMF_bt_vala_btca_01xC") | (df['layer'] == "EMF_bt_vala_btca_02xC") | (df['layer'] == "EMF_bt_vala_btca_03xC") | (df['layer'] == "EMF_bt_vala_btca_04xC") | (df['layer'] == "EMF_bt_vala_btca_05xC") | (df['layer'] == "EMF_bt_vala_btca_06xC") | (df['layer'] == "EMF_bt_vala_btca_07xC") | (df['layer'] == "EMF_bt_vala_btca_08xC")

mask_ac_watercourse = (df['layer'] == "EMF_bt_watercourse_ductbank_01xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_02xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_03xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_04xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_05xC") | (df['layer'] == "EMF_bt_watercourse_ductbank_07xC")

mask_ac_access = (df['layer'] == 'EMF_bt_access_ductbank_01xC') | (df['layer'] == 'EMF_bt_access_ductbank_02xC') | (df['layer'] == 'EMF_bt_access_ductbank_03xC') | (df['layer'] == 'EMF_bt_access_ductbank_04xC') | (df['layer'] == 'EMF_bt_access_ductbank_05xC') | (df['layer'] == 'EMF_bt_access_ductbank_06xC') | (df['layer'] == 'EMF_bt_access_ductbank_07xC') | (df['layer'] == 'EMF_bt_access_ductbank_08xC')

mask_dc_cables = (df['layer'] == "EMF_cabo_solar_negativo") | (df['layer'] == "EMF_cabo_solar_positivo")

mask_dc_ducts = (df['layer'] == 'EMF_bt_eletroduto_1S') | (df['layer'] == 'EMF_bt_eletroduto_2S') | (df['layer'] == 'EMF_bt_eletroduto_3S')

mask_dc_trenches = ~(mask_dc_cables | mask_ac_cables | mask_ac_watercourse | mask_ac_access | mask_dc_ducts)

QTY. OF AC CABLES (m)
=

In [7]:
ac_cables_df = df[mask_ac_cables]
print(ac_cables_df)

            type                  layer handle       start_x        start_y  \
6858  LWPOLYLINE  EMF_bt_vala_btca_01xC   BD4D  36054.691468  129707.174963   
6859  LWPOLYLINE  EMF_bt_vala_btca_01xC   BD4E  33680.184272  129012.403336   
6860  LWPOLYLINE  EMF_bt_vala_btca_01xC   BD4F  33699.041658  128996.220070   
6861  LWPOLYLINE  EMF_bt_vala_btca_01xC   BD50  33730.541658  128996.220070   
6862  LWPOLYLINE  EMF_bt_vala_btca_01xC   BD51  33768.367272  128996.220070   
...          ...                    ...    ...           ...            ...   
7959  LWPOLYLINE  EMF_bt_vala_btca_01xC   C1B5  36235.282551  128949.521525   
7960  LWPOLYLINE  EMF_bt_vala_btca_01xC   C1B6  36213.241051  128949.521525   
7961  LWPOLYLINE  EMF_bt_vala_btca_01xC   C1B7  36197.482551  128948.521525   
7962  LWPOLYLINE  EMF_bt_vala_btca_01xC   C1B8  36235.282551  128949.521525   
7963  LWPOLYLINE  EMF_bt_vala_btca_01xC   C1B9  36159.699551  128775.159425   

             end_x          end_y      length  
685

In [18]:
ac_cables_group = ac_cables_df.groupby(by=['layer']).sum().reset_index()
print(ac_cables_group[['layer','length']])

                   layer        length
0  EMF_bt_vala_btca_01xC  11612.351487
1  EMF_bt_vala_btca_02xC   7535.069065
2  EMF_bt_vala_btca_03xC   5263.376076
3  EMF_bt_vala_btca_04xC   4043.333054
4  EMF_bt_vala_btca_05xC   3160.560417
5  EMF_bt_vala_btca_06xC   2192.055292
6  EMF_bt_vala_btca_07xC   1411.144949
7  EMF_bt_vala_btca_08xC   1724.205683


QTY. OF AC DUCTS IN WATERCOURSE (m and un.)
=

In [8]:
ac_watercourse_df = df[mask_ac_watercourse]
print(ac_watercourse_df)

            type                             layer handle       start_x  \
7206  LWPOLYLINE  EMF_bt_watercourse_ductbank_05xC   BEA9  33898.320236   
7207  LWPOLYLINE  EMF_bt_watercourse_ductbank_03xC   BEAA  34134.813770   
7208  LWPOLYLINE  EMF_bt_watercourse_ductbank_02xC   BEAB  34118.690121   
7209  LWPOLYLINE  EMF_bt_watercourse_ductbank_02xC   BEAC  35990.719166   
7210  LWPOLYLINE  EMF_bt_watercourse_ductbank_04xC   BEAD  35750.203205   
7252  LWPOLYLINE  EMF_bt_watercourse_ductbank_07xC   BED7  34951.955246   
7744  LWPOLYLINE  EMF_bt_watercourse_ductbank_04xC   C0DC  34448.490087   
7915  LWPOLYLINE  EMF_bt_watercourse_ductbank_01xC   C187  34524.825913   
7916  LWPOLYLINE  EMF_bt_watercourse_ductbank_01xC   C188  34395.251598   
7917  LWPOLYLINE  EMF_bt_watercourse_ductbank_07xC   C189  35281.404466   
7918  LWPOLYLINE  EMF_bt_watercourse_ductbank_03xC   C18A  35692.507638   
7919  LWPOLYLINE  EMF_bt_watercourse_ductbank_01xC   C18B  34826.193102   

            start_y     

In [19]:
ac_watercourse_group = ac_watercourse_df.groupby(by=['layer']).sum().reset_index()
print(ac_watercourse_group[['layer','length']])

                              layer     length
0  EMF_bt_watercourse_ductbank_01xC  65.981001
1  EMF_bt_watercourse_ductbank_02xC  44.042757
2  EMF_bt_watercourse_ductbank_03xC  44.005550
3  EMF_bt_watercourse_ductbank_04xC  44.000000
4  EMF_bt_watercourse_ductbank_05xC  22.903539
5  EMF_bt_watercourse_ductbank_07xC  44.004562


In [20]:
print(ac_watercourse_df['layer'].value_counts())

layer
EMF_bt_watercourse_ductbank_01xC    3
EMF_bt_watercourse_ductbank_03xC    2
EMF_bt_watercourse_ductbank_02xC    2
EMF_bt_watercourse_ductbank_04xC    2
EMF_bt_watercourse_ductbank_07xC    2
EMF_bt_watercourse_ductbank_05xC    1
Name: count, dtype: int64


QTY. OF AC DUCTS IN ACCESS (m and un.)
=

In [9]:
ac_access_df = df[mask_ac_access]
print(ac_access_df)

            type                        layer handle       start_x  \
7301  LWPOLYLINE  EMF_bt_access_ductbank_07xC   BF08  33777.884490   
7302  LWPOLYLINE  EMF_bt_access_ductbank_02xC   BF09  34572.291500   
7303  LWPOLYLINE  EMF_bt_access_ductbank_01xC   BF0A  34614.229804   
7304  LWPOLYLINE  EMF_bt_access_ductbank_02xC   BF0B  34604.303969   
7305  LWPOLYLINE  EMF_bt_access_ductbank_04xC   BF0C  34619.200996   
7306  LWPOLYLINE  EMF_bt_access_ductbank_01xC   BF0D  33985.892217   
7310  LWPOLYLINE  EMF_bt_access_ductbank_05xC   BF11  34614.227071   
7311  LWPOLYLINE  EMF_bt_access_ductbank_03xC   BF12  35063.559104   
7312  LWPOLYLINE  EMF_bt_access_ductbank_07xC   BF13  35126.149710   
7313  LWPOLYLINE  EMF_bt_access_ductbank_05xC   BF14  35403.226868   
7314  LWPOLYLINE  EMF_bt_access_ductbank_01xC   BF15  35133.373545   
7315  LWPOLYLINE  EMF_bt_access_ductbank_07xC   BF16  35139.376003   
7316  LWPOLYLINE  EMF_bt_access_ductbank_01xC   BF17  35151.009070   
7318  LWPOLYLINE  EM

In [21]:
ac_access_group = ac_access_df.groupby(by=['layer']).sum().reset_index()
print(ac_access_group[['layer','length']])

                         layer     length
0  EMF_bt_access_ductbank_01xC  75.325145
1  EMF_bt_access_ductbank_02xC  45.619963
2  EMF_bt_access_ductbank_03xC  31.175095
3  EMF_bt_access_ductbank_04xC  28.202538
4  EMF_bt_access_ductbank_05xC  42.000006
5  EMF_bt_access_ductbank_06xC  38.730659
6  EMF_bt_access_ductbank_07xC  27.352969
7  EMF_bt_access_ductbank_08xC  36.366803


In [22]:
print(ac_access_df['layer'].value_counts())

layer
EMF_bt_access_ductbank_01xC    8
EMF_bt_access_ductbank_05xC    7
EMF_bt_access_ductbank_02xC    5
EMF_bt_access_ductbank_03xC    5
EMF_bt_access_ductbank_08xC    5
EMF_bt_access_ductbank_07xC    4
EMF_bt_access_ductbank_06xC    4
EMF_bt_access_ductbank_04xC    3
Name: count, dtype: int64


QTY. OF DC CABLES (m)
=

In [10]:
dc_cables_df = df[mask_dc_cables]
print(dc_cables_df)

             type                    layer handle      start_x        start_y  \
14654  LWPOLYLINE  EMF_cabo_solar_positivo   DBEE  36016.90525  129793.609288   
14655  LWPOLYLINE  EMF_cabo_solar_positivo   DBEF  36016.90525  129751.107288   
14656  LWPOLYLINE  EMF_cabo_solar_positivo   DBF0  36023.20525  129793.609288   
14657  LWPOLYLINE  EMF_cabo_solar_positivo   DBF1  36023.20525  129751.107288   
14658  LWPOLYLINE  EMF_cabo_solar_positivo   DBF2  36029.50525  129793.609288   
...           ...                      ...    ...          ...            ...   
40732  LWPOLYLINE  EMF_cabo_solar_negativo  141CC  34400.08615  126040.796699   
40733  LWPOLYLINE  EMF_cabo_solar_negativo  141CD  34400.08615  125998.294699   
40734  LWPOLYLINE  EMF_cabo_solar_negativo  141CE  34406.38615  126040.796699   
40735  LWPOLYLINE  EMF_cabo_solar_negativo  141CF  34406.38615  125998.294699   
40736  LWPOLYLINE  EMF_cabo_solar_negativo  141D0  34412.68615  125998.294699   

             end_x         

In [33]:
dc_cables_group = dc_cables_df.groupby(by=['layer']).sum().reset_index()
print(dc_cables_group[['layer','length']])
print(f'\nThe total length of DC cables is {round(dc_cables_group['length'].sum(),2)} meters')

                     layer        length
0  EMF_cabo_solar_negativo  1.069016e+06
1  EMF_cabo_solar_positivo  9.798205e+05

The total length of DC cables is 2048836.31 meters


QTY. OF DC DUCTS (m and un.)
=

In [11]:
dc_ducts_df =  df[mask_dc_ducts]
print(dc_ducts_df)

             type                 layer handle      start_x        start_y  \
10508  LWPOLYLINE  EMF_bt_eletroduto_2S   CBBC  36016.90525  129710.071688   
10509  LWPOLYLINE  EMF_bt_eletroduto_2S   CBBD  36023.20525  129710.071688   
10510  LWPOLYLINE  EMF_bt_eletroduto_2S   CBBE  36029.50525  129710.071688   
10511  LWPOLYLINE  EMF_bt_eletroduto_2S   CBBF  36035.80525  129710.071688   
10512  LWPOLYLINE  EMF_bt_eletroduto_2S   CBC0  36042.10525  129710.071688   
...           ...                   ...    ...          ...            ...   
33900  LWPOLYLINE  EMF_bt_eletroduto_1S  1271C  34242.59000  125998.282200   
33901  LWPOLYLINE  EMF_bt_eletroduto_1S  1271D  34248.89000  125998.282200   
33902  LWPOLYLINE  EMF_bt_eletroduto_1S  1271E  34255.19000  125998.282200   
33903  LWPOLYLINE  EMF_bt_eletroduto_1S  1271F  34261.49000  125998.282200   
33904  LWPOLYLINE  EMF_bt_eletroduto_1S  12720  34412.68615  125998.294699   

             end_x          end_y     length  
10508  36054.172

In [27]:
dc_ducts_group = dc_ducts_df.groupby(by=['layer']).sum().reset_index()
print(dc_ducts_group[['layer','length']])

                  layer        length
0  EMF_bt_eletroduto_1S  41748.203962
1  EMF_bt_eletroduto_2S  31690.800648
2  EMF_bt_eletroduto_3S  36640.672499


In [34]:
print(dc_ducts_df['layer'].value_counts())
print(f'\nThe total quantity of DC ducts is {len(dc_ducts_df)}')

layer
EMF_bt_eletroduto_3S    2748
EMF_bt_eletroduto_2S    1378
EMF_bt_eletroduto_1S    1368
Name: count, dtype: int64

The total quantity of DC ducts is 5494


QTY. OF DC TRENCHES (m)
=

In [12]:
dc_trenches_df = df[mask_dc_trenches]
print(dc_trenches_df)

             type            layer handle      start_x        start_y  \
0      LWPOLYLINE   EMF_vala_CC_2C   A283  34683.58615  126169.995377   
1      LWPOLYLINE   EMF_vala_CC_4C   A284  34685.28295  126171.419048   
2      LWPOLYLINE   EMF_vala_CC_1C   A285  34757.49320  126242.075025   
3      LWPOLYLINE   EMF_vala_CC_3C   A286  34536.99320  126293.210025   
4      LWPOLYLINE   EMF_vala_CC_2C   A287  34798.68295  126336.315931   
...           ...              ...    ...          ...            ...   
10503  LWPOLYLINE  EMF_vala_CC_20C   CBB4  34870.89320  126773.004650   
10504  LWPOLYLINE   EMF_vala_CC_7C   CBB5  34458.48295  126124.997603   
10505  LWPOLYLINE  EMF_vala_CC_21C   CBB6  34823.88680  126773.004200   
10506  LWPOLYLINE  EMF_vala_CC_15C   CBB7  34458.48295  126045.969514   
10507  LWPOLYLINE  EMF_vala_CC_23C   CBB8  34864.59320  126773.004650   

              end_x          end_y    length  
0      34685.282950  126171.419048  2.214942  
1      34689.886150  126171.1

In [36]:
dc_trenches_group = dc_trenches_df.groupby(by=['layer']).sum().reset_index()
print(dc_trenches_group[['layer','length']])
print(f'\nThe total length of DC trenches is {round(dc_trenches_group['length'].sum(),2)} meters')

              layer       length
0   EMF_vala_CC_10C  1440.649874
1   EMF_vala_CC_11C   970.922478
2   EMF_vala_CC_12C  2135.308464
3   EMF_vala_CC_13C   583.526674
4   EMF_vala_CC_14C   406.086868
5   EMF_vala_CC_15C   265.987412
6   EMF_vala_CC_16C    52.206284
7   EMF_vala_CC_17C    64.544916
8   EMF_vala_CC_18C    12.213818
9   EMF_vala_CC_19C     4.074550
10   EMF_vala_CC_1C  3694.786301
11  EMF_vala_CC_20C     6.300000
12  EMF_vala_CC_21C     4.074550
13  EMF_vala_CC_23C     4.074392
14   EMF_vala_CC_2C  5324.682607
15   EMF_vala_CC_3C  7273.210962
16   EMF_vala_CC_4C  4043.126396
17   EMF_vala_CC_5C  1963.380855
18   EMF_vala_CC_6C  3588.523193
19   EMF_vala_CC_7C  1465.702615
20   EMF_vala_CC_8C  1943.108761
21   EMF_vala_CC_9C  2799.308538

The total length of DC trenches is 38045.8 meters


OVERVIEW
=

In [39]:
with pd.ExcelWriter(r"C:\Users\Usuario\Desktop\BOQ.xlsx") as xlsx:
    ac_cables_group[['layer','length']].to_excel(xlsx, sheet_name='AC Trenches (m)', index=False)
    ac_watercourse_group[['layer','length']].to_excel(xlsx, sheet_name='AC Ducts Watercouse (m)', index=False)
    ac_watercourse_df['layer'].value_counts().to_excel(xlsx, sheet_name='AC Ducts Watercouse (un.)', index=False)
    ac_access_group[['layer','length']].to_excel(xlsx, sheet_name='AC Ducts Access (m)', index=False)
    ac_access_df['layer'].value_counts().to_excel(xlsx, sheet_name='AC Ducts Access (un.)', index=False)
    dc_cables_group[['layer','length']].to_excel(xlsx, sheet_name='DC Cables (m)', index=False)
    dc_ducts_group[['layer','length']].to_excel(xlsx, sheet_name='DC Ducts (m)', index=False)
    dc_ducts_df['layer'].value_counts().to_excel(xlsx, sheet_name='DC Ducts (un.)', index=False)
    dc_trenches_group[['layer','length']].to_excel(xlsx, sheet_name='DC Trenches (un.)', index=False)
